# Other Visualizations: Model Evaluation Diagnostics

These visualizations are for comparing model predictions against ground truth.
They apply to any PDE-based or learned operator model, not specifically to barotropic data.

Requires a prediction field and a truth field of the same shape.

| # | Visualization | What it shows |
|---|---|---|
| 3.1 | Error Maps | Spatial error structure (absolute, signed, relative) |
| 3.2 | Power Spectrum Comparison | Spectral fidelity of predictions |
| 3.3 | PDE Residual Map | Physics consistency of model output |

---
## Setup: Synthetic Prediction Data

Since we do not have model predictions yet, we generate a synthetic prediction
by Gaussian-blurring the true vorticity field. This mimics the over-smoothing
bias common in learned operators.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
from scipy.ndimage import gaussian_filter

plt.rcParams["figure.dpi"] = 120

def preprocess_dataset(ds):
    for v in ds.variables:
        if "units" in ds[v].attrs and "0000-00-00" in ds[v].attrs["units"]:
            ds[v].attrs["units"] = ds[v].attrs["units"].replace("0000-00-00", "0001-01-01")
        if ds[v].attrs.get("calendar") == "NO_CALENDAR":
            ds[v].attrs["calendar"] = "360_day"
    return ds

ds = xr.open_mfdataset(
    "../output/barotropic_stirring-T85/run*/atmos_daily.nc",
    combine="by_coords",
    data_vars="minimal",
    coords="minimal",
    compat="override",
    decode_times=False,
    preprocess=preprocess_dataset,
)
ds = xr.decode_cf(ds, decode_times=xr.coders.CFDatetimeCoder(use_cftime=True), decode_timedelta=True)

lon = ds.lon.values
lat = ds.lat.values
lat_rad = np.deg2rad(lat)
cos_weights = np.cos(lat_rad).clip(0)
EARTH_RADIUS = 6.371e6

rng = np.random.default_rng(42)

# Use first time step as "truth", blur it as "prediction"
VOR_TRUE = ds.vor.isel(time=0).values
VOR_PRED = gaussian_filter(VOR_TRUE, sigma=1.5) + 0.08 * rng.standard_normal(VOR_TRUE.shape) * VOR_TRUE.std()

print(f"Loaded: {dict(ds.sizes)}")
print(f"Truth range: {VOR_TRUE.min():.2e} to {VOR_TRUE.max():.2e}")
print(f"Pred range:  {VOR_PRED.min():.2e} to {VOR_PRED.max():.2e}")

---
## 3.1 Error Maps

Three views of the error between predicted and true vorticity:

- **Absolute error** $|\hat{\zeta} - \zeta|$: Where is the error largest?
- **Signed error** $\hat{\zeta} - \zeta$: Over-prediction (red) vs under-prediction (blue).
- **Relative error** $|\hat{\zeta} - \zeta| / |\zeta|$: Error relative to local signal strength.

Always look at spatial error maps before relying on scalar metrics like RMSE --
a model with low RMSE might still have large localized errors.

In [ ]:
abs_error = np.abs(VOR_PRED - VOR_TRUE)
signed_error = VOR_PRED - VOR_TRUE
rel_error = abs_error / (np.abs(VOR_TRUE) + 1e-10 * VOR_TRUE.std())

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

vlim = np.percentile(abs_error, 98)
im0 = axes[0].pcolormesh(lon, lat, abs_error, cmap="Reds", vmin=0, vmax=vlim, shading="auto")
fig.colorbar(im0, ax=axes[0], label="|error| [s^-1]")
axes[0].set_title("Absolute Error")

vlim = np.percentile(np.abs(signed_error), 98)
im1 = axes[1].pcolormesh(lon, lat, signed_error, cmap="RdBu_r", vmin=-vlim, vmax=vlim, shading="auto")
fig.colorbar(im1, ax=axes[1], label="Signed error [s^-1]")
axes[1].set_title("Signed Error")

vlim = np.percentile(rel_error, 95)
im2 = axes[2].pcolormesh(lon, lat, rel_error, cmap="YlOrRd", vmin=0, vmax=vlim, shading="auto")
fig.colorbar(im2, ax=axes[2], label="Relative error")
axes[2].set_title("Relative Error")

for ax in axes:
    ax.set_xlabel("Longitude [deg]")
    ax.set_ylabel("Latitude [deg]")

plt.suptitle("Error Maps: Prediction vs Truth", y=1.02)
plt.tight_layout()
plt.show()

---
## 3.2 Power Spectrum Comparison

**Left:** Zonal power spectrum of both fields on log-log axes.
**Right:** Spectral ratio $P_{\text{pred}}(k) / P_{\text{true}}(k)$.

Ratio < 1 = prediction missing energy (over-smoothed).
Ratio > 1 = excess energy (noisy).
Ratio = 1 = spectrally accurate.

In [ ]:
def power_spectrum(field, weights):
    f_hat = np.fft.rfft(field, axis=1)
    power = (np.abs(f_hat) ** 2) * weights[:, None]
    return power.sum(axis=0) / weights.sum()

P_true = power_spectrum(VOR_TRUE, cos_weights)
P_pred = power_spectrum(VOR_PRED, cos_weights)
k = np.arange(len(P_true))
k[0] = 1
K_MAX = 40

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].loglog(k[1:K_MAX], P_true[1:K_MAX], label="True")
axes[0].loglog(k[1:K_MAX], P_pred[1:K_MAX], label="Prediction", linestyle="--")
axes[0].set_xlabel("Zonal wavenumber k")
axes[0].set_ylabel("Power")
axes[0].set_title("Power Spectra")
axes[0].legend()

ratio = P_pred[1:K_MAX] / (P_true[1:K_MAX] + 1e-40)
axes[1].semilogx(k[1:K_MAX], ratio)
axes[1].axhline(1.0, color="k", linestyle="--", linewidth=0.8)
axes[1].set_xlabel("Zonal wavenumber k")
axes[1].set_ylabel("Pred / True power ratio")
axes[1].set_title("Spectral Ratio")
axes[1].set_ylim(0, 2)

plt.suptitle("Power Spectrum Comparison")
plt.tight_layout()
plt.show()

---
## 3.3 PDE Residual Map

Residual of the barotropic vorticity equation evaluated from model output:

$$R = \frac{\partial\zeta}{\partial t} + \mathbf{u} \cdot \nabla\zeta$$

If the output satisfies the PDE exactly, $R = 0$ everywhere.
Non-zero residual reveals where the model violates the governing physics.

Computed between consecutive time steps using finite differences.

In [ ]:
dlat = lat_rad[1] - lat_rad[0]
dlon = np.deg2rad(lon[1] - lon[0])
cos_lat_2d = np.cos(np.deg2rad(lat))[:, None] * np.ones((1, len(lon)))
cos_lat_2d = cos_lat_2d.clip(1e-6)

# Use first two time steps
vor0 = ds.vor.isel(time=0).values
vor1 = ds.vor.isel(time=1).values
u0 = ds.ucomp.isel(time=0).values
v0 = ds.vcomp.isel(time=0).values

# Time derivative (finite difference) -- assume 1-day intervals
DT = 86400.0
dvdt = (vor1 - vor0) / DT

# Advection
dvor_dx = np.gradient(vor0, dlon, axis=1) / (EARTH_RADIUS * cos_lat_2d)
dvor_dy = np.gradient(vor0, dlat, axis=0) / EARTH_RADIUS
advection = u0 * dvor_dx + v0 * dvor_dy
residual = dvdt + advection

fig, ax = plt.subplots(figsize=(12, 5))
vlim = np.percentile(np.abs(residual), 97)
im = ax.pcolormesh(lon, lat, residual, cmap="RdBu_r", vmin=-vlim, vmax=vlim, shading="auto")
fig.colorbar(im, ax=ax, label="Residual [s^-2]")
ax.set_title("PDE Residual: d(zeta)/dt + u.grad(zeta)")
ax.set_xlabel("Longitude [deg]")
ax.set_ylabel("Latitude [deg]")
plt.tight_layout()
plt.show()

---
## Summary

| # | Visualization | What it shows |
|---|---|---|
| 3.1 | Error Maps | Spatial error structure (abs/signed/relative) |
| 3.2 | Power Spectrum | Spectral fidelity of predictions |
| 3.3 | PDE Residual | Physics consistency of model output |

These diagnostics complement the barotropic-specific visualizations in
`barotropic_visualization.ipynb`.